In [ ]:
import os
os.environ["PATH"] += ":/mnt/large-disk/guidance-team/edm2-guidance/bin"

In [ ]:
!PYTHONPATH=./:./base:./classifer:./edm2impl torchrun --standalone --nproc_per_node=3 ./edm2impl/generate_images_for_test.py \
  --preset=edm2-img512-s-guid-fid \
  --outdir=/mnt/large-disk/guidance-team/outCFG \
  --subdirs \
  --seeds=0-50010 \
    --debug=False \
--sampler=random_ddg \
--guidance_scheduler=interval_scheduler \
--guidance=2.1 \
--k_average=4 \
--negative_class_csv_path=./plot/negative_class_manual.csv \
debug=True


In [ ]:
from PIL import Image
import os

def check_images(dir_path):
    broken_files = []
    for root, _, files in os.walk(dir_path):
        for name in files:
            path = os.path.join(root, name)
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception as e:
                print(f"[BROKEN] {path} -> {e}")
                broken_files.append(path)
    return broken_files

broken = check_images("/mnt/large-disk/guidance-team/outCFG")
print(f"Found {len(broken)} broken images.")


In [ ]:
!PYTHONPATH=./:./base:./classifer:./edm2impl torchrun --standalone --nproc_per_node=1 ./base/calculate_metrics.py calc \
--images=/mnt/large-disk/guidance-team/outCFG \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl \
--num=10000 \
--metrics='fid' \
--batch=4


In [ ]:
import torch
from base.torch_utils import distributed as dist
import sys
import tqdm

sys.path.append('./base')
sys.path.append('./classifer')
sys.path.append('./edm2impl')
from base.calculate_metrics import load_stats,calculate_stats_for_files,calculate_metrics_from_stats
def run_calc(image_path, ref_path, metrics, num_images, max_batch_size, num_workers):
    torch.multiprocessing.set_start_method('spawn')
    dist.init()
    if dist.get_rank() == 0:
        ref = load_stats(path=ref_path)
    stats_iter = calculate_stats_for_files(image_path=image_path, metrics=metrics,
                                           num_images=num_images, max_batch_size=max_batch_size,
                                           num_workers=num_workers)
    for r in tqdm.tqdm(stats_iter, unit='batch', disable=(dist.get_rank() != 0)):
        pass
    if dist.get_rank() == 0:
        return calculate_metrics_from_stats(stats=r.stats, ref=ref, metrics=metrics)
    torch.distributed.barrier()


In [ ]:
run_calc(
    image_path='/mnt/large-disk/guidance-team/outCFG',
    ref_path='https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img512.pkl',
    metrics=['fid'],#, 'fd_dinov2'],
    num_images=10000,
    max_batch_size=1,
    num_workers=2
)
